# 期末專題報告 61275076H 王俐璇
🏞️ AI 旅遊嚮導：結合圖文生成的個人化旅遊推薦系統
- 詳細說明皆在PDF文件中

In [ ]:
# 📌 安裝必要套件
!pip install openai
!pip install openai folium gradio
!pip install openai requests
# 🚀下載 stabilityai/stable-diffusion-2-1 模型
# 直接在本地模型生成圖片
!pip install diffusers accelerate transformers safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3

In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
from PIL import Image
import gradio as gr
import requests
import time

In [ ]:
# 📌 Groq 登入
for var in ["PROVIDER", "GROQ_API_KEY", "OPENAI_API_KEY"]:
    if var in os.environ:
        os.environ.pop(var)
        print(f"🔄 已清空環境變數: {var}")

# 輸入 Groq API 金鑰
os.environ["GROQ_API_KEY"] = input("請輸入 Groq API 金鑰 (gsk- 開頭)：").strip()

api_key = os.environ.get("GROQ_API_KEY")
model = "meta-llama/llama-4-scout-17b-16e-instruct"
api_url = "https://api.groq.com/openai/v1/chat/completions"

# 檢查金鑰是否存在
if not api_key:
    raise ValueError("❌ 找不到 Groq API 金鑰，請先設定環境變數！")

print(f"✅ 使用模型：GROQ ({model})")
print(f"✅ API Endpoint：{api_url}")

In [ ]:
# ✅ Groq API 金鑰驗證

def check_groq_api_key(api_key, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    api_url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": "Say hello"}],
        "max_tokens": 5
    }

    try:
        response = requests.post(api_url, headers=headers, json=payload, timeout=10)
        if response.status_code == 200:
            print("✅ Groq API 金鑰驗證成功！可以正常使用！")
            print("✅ 測試回應:", response.json()["choices"][0]["message"]["content"])
            return True
        else:
            print(f"❌ Groq API 錯誤！狀態碼: {response.status_code}")
            print("錯誤訊息:", response.text)
            return False
    except Exception as e:
        print(f"❌ 發生例外錯誤: {e}")
        return False

# 金鑰驗證
check_groq_api_key(api_key)

✅ Groq API 金鑰驗證成功！可以正常使用！
✅ 測試回應: Hello! It's nice to


True

In [ ]:
# ✅ LLM (Groq API)回應
def llm_reply(prompt, chat_history=None):
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    # 支援多輪聊天，傳遞過去歷史訊息
    messages = chat_history if chat_history else []
    messages.append({"role": "user", "content": prompt})
    body = {
        "model": model,
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 300
    }
    try:
        response = requests.post(api_url, headers=headers, json=body)
        if response.status_code == 200:
            reply = response.json()["choices"][0]["message"]["content"]
            print(f"✅ LLM 回應: {reply}")
            messages.append({"role": "assistant", "content": reply})
            return reply, messages
        else:
            print(f"❌ Groq API 錯誤: {response.status_code} - {response.text}")
            return f"❌ Groq API 錯誤: {response.status_code}", messages
    except Exception as e:
        print(f"❌ 發生例外錯誤: {e}")
        return f"❌ 發生例外錯誤: {e}", messages

In [ ]:
# 📌 Diffusion 模型初始化
pipe = None

def load_diffusion_model():
    global pipe
    if pipe is None:
        print("⏳ 正在加載 Diffusion 模型...")
        model_id = "stabilityai/stable-diffusion-2-1"
        scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            scheduler=scheduler,
            torch_dtype=torch.float16
        )
        pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
        print("✅ 模型加載完成！")
    else:
        print("✅ 模型已經加載，直接使用！")
    return pipe

In [ ]:
STYLE_PRESETS = {
    "手繪風": "hand-drawn style, illustration, warm color palette, soft warm tones, high detail",
    "日系可愛": "anime style, cute, Japanese art, pastel colors, soft light",
    "寫實風": "realistic style, high detail, cinematic lighting, 4k resolution",
    "像素風": "pixel art, 8-bit style, retro game aesthetics, bright colors",
    "水彩風": "watercolor painting, soft edges, delicate brush strokes, muted colors"
}

In [ ]:
# 📌 Diffusion 圖像生成
DEFAULT_STYLE_KEY = "手繪風"

def generate_cover_image(prompt, style_choice, default_key=DEFAULT_STYLE_KEY):
    STYLE_PRESETS = {
        "手繪風": "hand-drawn style, illustration, warm color palette, soft warm tones, high detail",
        "日系可愛": "anime style, cute, Japanese art, pastel colors, soft light",
        "寫實風": "realistic style, high detail, cinematic lighting, 4k resolution",
        "像素風": "pixel art, 8-bit style, retro game aesthetics, bright colors",
        "水彩風": "watercolor painting, soft edges, delicate brush strokes, muted colors"
    }
    try:
        load_diffusion_model()
        if style_choice in STYLE_PRESETS:
            style = STYLE_PRESETS[style_choice]
            print(f"🎨 使用者選擇風格：「{style_choice}」")
        else:
            style = STYLE_PRESETS[default_key]
            print(f"🎨 使用者未選擇風格或選擇無效，已套用預設風格：「{default_key}」")
        full_prompt = f"travel guide cover, {prompt}, {style}"
        print(f"🎨 開始生成圖片：{full_prompt}")
        start_time = time.time()
        result = pipe(full_prompt, num_inference_steps=30, guidance_scale=7.5)
        image = result.images[0]
        duration = time.time() - start_time
        print(f"✅ 圖片生成完成，用時 {duration:.1f} 秒")
        return image
    except Exception as e:
        print(f"❌ 發生例外錯誤：{e}")
        return Image.new("RGB", (512, 512), color="gray")

In [ ]:
# 📌 AI Agents - Reflection & Planning Prompt
def agent_plan_route(location, preference, budget, days, group, transport, season="自動推斷"):
    # Reflection + Planning Prompt
    prompt = (
        f"你是資深旅遊規劃AI，具備專業知識與即時判斷能力。\n"
        f"【旅遊資訊】\n"
        f"地點：{location}\n"
        f"偏好：{preference}\n"
        f"預算：{budget}\n"
        f"天數：{days}\n"
        f"人數/身份：{group}\n"
        f"交通方式：{transport}\n"
        f"時節/季節：{season}\n"
        f"請針對上述需求，進行：\n"
        f"1. 完整行程自動規劃，並列出每日路線\n"
        f"2. Reflection：檢查有無「明顯不合理」安排（如冬天安排賞櫻、交通中斷等），自動修正，並說明調整理由\n"
        f"3. 行程包含天氣/景點/交通/餐飲等推薦，禁止出現無法實現的內容\n"
        f"4. 以條列式輸出"
    )
    result, _ = llm_reply(prompt)
    return result

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🌏 AI旅遊嚮導：結合圖文生成的個人化旅遊推薦系統")
    with gr.Tab("個人化旅遊推薦"):
        with gr.Row():
            location = gr.Textbox(label="旅遊地點/主題", placeholder="東京鐵塔、京都櫻花")
            preference = gr.Textbox(label="旅遊偏好", placeholder="自然景觀/人文藝術/美食/購物")
            budget = gr.Textbox(label="預算（可選）", placeholder="例如：2萬元以內")
            days = gr.Textbox(label="天數（可選）", placeholder="例如：5天4夜")
            group = gr.Textbox(label="人數/身份（可選）", placeholder="2人情侶/全家出遊")
            transport = gr.Textbox(label="交通方式（可選）", placeholder="自駕/大眾運輸")
            style_choice = gr.Radio(
                choices=list(STYLE_PRESETS.keys()),
                label="圖像風格（請選擇一項）",
                value="手繪風"
            )
        with gr.Row():
            submit = gr.Button("生成旅遊建議、圖像與 AI Agents 行程")
        llm_output = gr.Textbox(label="旅遊建議（RAG+LLM推薦景點）", lines=5)
        image_output = gr.Image(label="專屬旅遊場景圖片")
        agent_output = gr.Textbox(label="AI Agents 行程規劃 (Reflection & Planning)", lines=10)
        def ai_travel_assistant_all(location, preference, budget, days, group, transport, style_choice):
            # LLM推薦
            user_prompt = (
                f"請根據以下資訊規劃個人化旅遊建議，包含路線/景點/活動。\n"
                f"地點：{location}\n"
                f"旅遊偏好：{preference}\n"
                f"預算：{budget}\n"
                f"天數：{days}\n"
                f"人數或身份：{group}\n"
                f"交通方式：{transport}\n"
                f"請條列化建議內容。"
            )
            llm_result, _ = llm_reply(user_prompt)
            image_result = generate_cover_image(location, style_choice)
            # AI Agents (Reflection + Planning)
            agent_result = agent_plan_route(location, preference, budget, days, group, transport)
            return llm_result, image_result, agent_result
        submit.click(
            fn=ai_travel_assistant_all,
            inputs=[location, preference, budget, days, group, transport, style_choice],
            outputs=[llm_output, image_output, agent_output]
        )

    with gr.Tab("互動式旅遊聊天機器人"):
        chatbox = gr.Chatbot(label="旅遊小助手：自由提問、推薦、查詢、規劃皆可")
        chat_input = gr.Textbox(label="請輸入你的問題", placeholder="請問大阪春天有什麼活動？")
        chat_submit = gr.Button("發送")
        # 保持聊天歷史（用state）
        chat_state = gr.State([])  # List of messages

        def chat_ai(user_msg, chat_history):
            # chat_history 格式：[{"role":"user/assistant","content":...}, ...]
            reply, new_history = llm_reply(user_msg, chat_history)
            chat_history.append((user_msg, reply))
            return chat_history, new_history

        chat_submit.click(
            fn=chat_ai,
            inputs=[chat_input, chat_state],
            outputs=[chatbox, chat_state]
        )

demo.launch()

<ipython-input-21-c50add219ce6>:45: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbox = gr.Chatbot(label="旅遊小助手：自由提問、推薦、查詢、規劃皆可")


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75c919d8c9281e93b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Hugging Face 簡易圖片生成（測試版）


In [ ]:
import requests
import base64
from PIL import Image
from io import BytesIO

# ✅ 設定你的 Hugging Face Token
HF_TOKEN = input("請輸入 Hugging Face Token (hf- 開頭)：").strip()

# ✅ 使用支援 API 的模型
HF_SD_MODEL = "stabilityai/stable-diffusion-2"
api_url = f"https://api-inference.huggingface.co/models/{HF_SD_MODEL}"

# ✅ 要畫的圖片描述
prompt = "富士山不僅是日本的象徵，亦是著名的旅遊景點和登山勝地"

# ✅ 發送請求
headers = {"Authorization": f"Bearer {HF_TOKEN}"}
payload = {"inputs": prompt}

response = requests.post(api_url, headers=headers, json=payload)
if response.status_code == 200:
    image_bytes = response.content
    image = Image.open(BytesIO(image_bytes))
    image.show()
else:
    print(f"❌ API 錯誤！狀態碼: {response.status_code}")
    print(response.text)

In [ ]:
!pip install diffusers accelerate transformers safetensors

import torch
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler

model_id = "stabilityai/stable-diffusion-2-1"  # 改用 2-1，支援度高
scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")

pipe = StableDiffusionPipeline.from_pretrained(model_id, scheduler=scheduler, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

# 渲染圖像
prompt = "富士山不僅是日本的象徵，亦是著名的旅遊景點和登山勝地"
image = pipe(prompt).images[0]

# 顯示圖片
image.show()

# 存檔
image.save("fuji_mountain.png")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.46G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]